# Ridge regression (L2 regularization)

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
When a model overfits, its coefficients become huge (say +8000 and −7990) so the curve can pass through every point. Ridge adds a **penalty** on large coefficients to the cost:

$$J(\mathbf w)=\underbrace{\sum_i(y_i-\hat y_i)^2}_{\text{error}}+\alpha\underbrace{\sum_j w_j^2}_{\text{L2 penalty}}$$

- $\alpha=0$: ordinary linear regression.
- Large $\alpha$: coefficients shrink toward zero and the curve becomes smoother.
- Very large $\alpha$: all coefficients are nearly zero and the model just predicts the mean (underfitting).

> The intercept $b$ is not penalized.

## The closed formula
$$\mathbf w^*=(X^\top X+\alpha I)^{-1}X^\top\mathbf y$$

Adding $\alpha I$ makes the matrix always invertible. That is why Ridge is also a good cure for **multicollinearity**.

## The coefficient path
As $\alpha$ grows, all coefficients shrink **smoothly**, but none becomes exactly zero:

> [Ridge coefficient paths on the diabetes data.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/ridge_path.png)

*Ridge coefficient paths on the diabetes data.*

## Lab
See a degree-12 polynomial with different values of $\alpha$. When $\alpha$ is tiny the curve swings wildly and the coefficients (bar chart) are large:

[[lab:reg_ridge]]

## Scaling is required
The penalty $\sum w_j^2$ treats all coefficients the same. If one feature is in meters and another in millimeters, the penalty is unfair. So always use `StandardScaler` before Ridge.



In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(1)
x = rng.uniform(0, 1, 20)[:, None]
y = np.sin(2 * np.pi * x.ravel()) + rng.normal(0, 0.2, 20)

for name, reg in [("Linear", LinearRegression()), ("Ridge a=0.001", Ridge(0.001)), ("Ridge a=1", Ridge(1)), ("Ridge a=100", Ridge(100))]:
    m = make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), reg).fit(x, y)
    w = m[-1].coef_
    print(f"{name:14s} largest |w| = {abs(w).max():12.2f}   sum of w^2 = {np.sum(w ** 2):14.2f}")



## Choosing α by cross-validation
`RidgeCV` tries several values of α with cross-validation:



In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 30))).fit(X_tr, y_tr)
print("Chosen alpha:", round(model[-1].alpha_, 3))
print("Test R2:", round(model.score(X_te, y_te), 3))



| Hyperparameter | Effect | Typical start |
|---|---|---|
| `alpha` | Strength of the penalty | 0.1 to 100; search on a log scale |
| `solver` | How it is solved | `auto` |

## Compared with linear regression
| | Linear | Ridge |
|---|---|---|
| Overfitting with many features | High | Lower |
| Multicollinearity | Unstable coefficients | Stable coefficients |
| Feature selection | No | No (coefficients do not become zero) |
| Needs scaling | No | **Yes** |

### Exercise

Implement Ridge with the closed formula on the scaled diabetes data and compare with scikit-learn.

In [ ]:
# Your answer here